# Retrieval baseline

Eight original demo passages and questions check the retrieval plumbing. These are authored diagnostics, not an independent ministry-data evaluation. No private book text is included.

In [1]:
# Load the local tools
import json
import sys
import tempfile
from pathlib import Path

root = Path.cwd()
if not (root / "search.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
from search import SearchIndex, build_index
from evaluate import evaluate
temporary = tempfile.TemporaryDirectory()
index_path = Path(temporary.name) / "demo.sqlite"
summary = build_index([root / "examples"], index_path)
print("Demo chunks:", summary["chunks"])
print("Import audit:", summary["audit"])

Demo chunks: 8
Import audit: {'source_rows': 8, 'conflicting_sections_excluded': 0, 'accepted_sections': 8}


In [2]:
# Measure title retrieval on the authored questions
questions = [json.loads(line) for line in (root / "examples/questions.jsonl").read_text().splitlines()]
result = evaluate(index_path, questions, "lexical", 4)
print("Queries:", result["queries"])
print("Hit at 4:", result["hit_at_k"])
print("Mean reciprocal rank:", result["mean_reciprocal_rank"])

Queries: 8
Hit at 4: 1.0
Mean reciprocal rank: 1.0


In [3]:
# Inspect a result instead of relying on the aggregate
hits = SearchIndex(index_path).search("Do citations prove every claim?", limit=3)
for rank, hit in enumerate(hits, 1):
    print(rank, hit["title"])
    print(hit["text"][:180])
print("Unmatched query:", SearchIndex(index_path).search("zyxwvut"))


1 Checking generated answers
Supply retrieved passages to the language model with numbered citations. The model should say when the passages do not support an answer. Validate that returned citation numbers we
Unmatched query: []


In [4]:
# Check archive import and conversational topic questions with original text
import html
from zipfile import ZipFile
from search import retrieval_question

archive_path = Path(temporary.name) / "original-demo.zip"
rows = [json.loads(line) for line in (root / "examples/sections.jsonl").read_text().splitlines()]
with ZipFile(archive_path, "w") as archive:
    for row in rows:
        title = row["book_title"]
        page = f"<h1>{html.escape(title)}</h1><p>{html.escape(row['text'])}</p>"
        archive.writestr(f"books/clean_html/{title}/page_001.html", page)
    archive.writestr("books/html_books/Extra_original_demo.html", "<h1>Extra original demo</h1><p>A whole-book export fills a missing page representation.</p>")
    archive.writestr("unrelated/private.txt", "This must not enter the index.")
archive_index = Path(temporary.name) / "archive.sqlite"
summary = build_index([archive_path], archive_index)
print("Archive demo books:", len(SearchIndex(archive_index).titles()))
print("Archive accepted pages:", summary["audit"]["accepted_sections"])
question = "What does the author say about citations?"
print("Retrieval topic:", retrieval_question(question))
hits = SearchIndex(archive_index).search(question, limit=1)
print("First title:", hits[0]["title"])
print("Export pages:", hits[0]["pages"])
temporary.cleanup()

Archive demo books: 9
Archive accepted pages: 9
Retrieval topic: citations
First title: Checking generated answers
Export pages: [1, 1]


In [5]:
# Check metadata routing and Bible HTML parsing with original diagnostics
from catalog import author_scope, has_author, title_key
from bible_html import read_bible
from search import verse_reference

assert author_scope("What does Brother Lee say about life?") == "Witness Lee"
assert author_scope("What does Watchman Nee say about life?") == "Watchman Nee"
assert author_scope("What does the ministry say about life?") == "all"
assert author_scope("Compare Witness Lee and Watchman Nee on life") == "all"
assert has_author("Watchman Nee and Witness Lee", "Witness Lee")
assert title_key("Knowledge of Life, The") == title_key("The Knowledge of Life")
assert verse_reference("Explain John 6:63") == "John 6:63"

with tempfile.TemporaryDirectory() as folder:
    bible = Path(folder)
    # These strings are parser fixtures, not Scripture or publisher commentary
    (bible / "Joh.htm").write_text("<b><a name=v6_63>Sample</a></b>Original parser diagnostic.<br><b><a name=v6_64a>Part A</a></b>First part.<q>Line break.</q><b><a name=v6_64b>Part B</a></b>Second part.</body>", encoding="utf-8")
    (bible / "JohN.htm").write_text("<a name=n6_63x1></a><p>Original explanatory fixture.</p>", encoding="utf-8")
    rows, files = read_bible(bible)
    assert len(rows) == 3 and len(files) == 2
    assert "First part." in rows[1]["text"] and "Second part." in rows[1]["text"]
    source = bible / "rows.jsonl"
    source.write_text("\n".join(json.dumps(row) for row in rows), encoding="utf-8")
    db_path = bible / "index.sqlite"
    build_index([source], db_path)
    hits = SearchIndex(db_path).search("John 6:63", collection="notes")
    assert hits and hits[0]["heading"] == "John 6:63 footnote 1"
    assert hits[0]["kind"] == "notes"
print("Author routing, joint authorship, split verses, footnote scope and exact references: passed")


Author routing, joint authorship, split verses, footnote scope and exact references: passed


Next compare lexical and hybrid retrieval on the same reviewed questions from a real collection. Track retrieval misses separately from unsupported generated answers. Title-level hits here do not measure passage-level recall or answer quality.